# License Plate Segmentation Pipeline

**Binary license plate segmentation** using the Simon Graves License Plate Dataset (Kaggle: `simongraves/license-plate-dataset`).

This notebook is **self-contained**. Run sections **in order**. Each major section ends with a **STOP — Review before continuing** cell. Do **not** skip those checks.

| Section | Topic |
|---------|--------|
| 0 | Environment setup |
| 1 | Dataset download |
| 2 | Dataset inspection (annotation type) |
| 3 | Mask generation (Case B/C only) |
| 4 | Dataset validation |
| 5 | Dataset class & DataLoader |
| 6 | Model (U-Net + ResNet34) |
| 7 | Loss, optimizer, metrics |
| 8 | Sanity check (2 mini-epochs) |
| 9 | Full training |
| 10 | Training curves |
| 11 | Evaluation |
| 12 | Final report template |

**Target:** binary mask (`0` = background, plate region = positive). Not object detection, not OCR.

## SECTION 0 — Environment Setup

Install missing packages, then verify CUDA and library versions.

In [ ]:
# Run this cell first — installs all required packages
import subprocess
import sys

packages = [
    "kagglehub[pandas-datasets]",
    "opencv-python",
    "albumentations",
    "segmentation-models-pytorch",
    "matplotlib",
    "pandas",
    "tqdm",
    "Pillow",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

In [ ]:
import sys
import torch
import torchvision
import cv2
import albumentations
import segmentation_models_pytorch as smp
import numpy as np
import pandas as pd

print(f"Python:          {sys.version}")
print(f"PyTorch:         {torch.__version__}")
print(f"torchvision:     {torchvision.__version__}")
print(f"OpenCV:          {cv2.__version__}")
print(f"Albumentations:  {albumentations.__version__}")
print(f"SMP:             {smp.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: CUDA is False. Training on GPU will not work.")

✅ **STOP — Section 0 complete.**

Verify that CUDA shows `True` and the GPU name is correct before continuing.

If CUDA is `False`, do NOT proceed to training.

## SECTION 1 — Dataset Download

Download `simongraves/license-plate-dataset` via **kagglehub** into `data/simongraves-license-plate-dataset/`.

Credentials are read from `~/.kaggle/kaggle.json` (or Windows equivalent). No username/API key is hardcoded.

In [ ]:
import shutil
from pathlib import Path

DATASET_LOCAL_PATH = Path("data/simongraves-license-plate-dataset")
DATASET_LOCAL_PATH.mkdir(parents=True, exist_ok=True)

try:
    import kagglehub

    cache_path = kagglehub.dataset_download("simongraves/license-plate-dataset")
    print(f"Kaggle cache path: {cache_path}")

    if not any(DATASET_LOCAL_PATH.iterdir()):
        shutil.copytree(cache_path, DATASET_LOCAL_PATH, dirs_exist_ok=True)
        print(f"Dataset copied to: {DATASET_LOCAL_PATH.resolve()}")
    else:
        print(f"Dataset already present at: {DATASET_LOCAL_PATH.resolve()}")

except Exception as exc:
    msg = str(exc).lower()
    if "credential" in msg or "kaggle.json" in msg or "unauthorized" in msg or "401" in msg:
        print("ERROR: Kaggle credentials not found.")
        print(r"Place kaggle.json at C:\Users\<YourName>\.kaggle\kaggle.json")
        print("Get your API token from: https://www.kaggle.com/settings/account")
    else:
        print(f"ERROR: Dataset download failed: {exc}")
        print("ERROR: Kaggle credentials not found.")
        print(r"Place kaggle.json at C:\Users\<YourName>\.kaggle\kaggle.json")
        print("Get your API token from: https://www.kaggle.com/settings/account")

✅ **STOP — Section 1 complete.**

Confirm the dataset path printed above is correct and contains files before continuing.

## SECTION 2 — Dataset Inspection

Confirm directory layout, TSV columns, and **annotation type** before generating any masks.

In [ ]:
# Cell 2.1 — Directory tree
from pathlib import Path
import collections

root = Path("data/simongraves-license-plate-dataset")

ext_counts = collections.Counter()
dir_counts = collections.defaultdict(int)

for f in root.rglob("*"):
    if f.is_file():
        ext_counts[f.suffix.lower()] += 1
        dir_counts[f.parent.name] += 1

print("=== File extensions ===")
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"  {ext:15s} {count}")

print("\n=== Files per directory ===")
for d, count in sorted(dir_counts.items(), key=lambda x: -x[1]):
    print(f"  {d:30s} {count}")

print(f"\nTotal files: {sum(ext_counts.values())}")

In [ ]:
# Cell 2.2 — Load the TSV annotation file
import pandas as pd
from pathlib import Path

root = Path("data/simongraves-license-plate-dataset")

tsv_path = next(root.rglob("*.tsv"), None)
if tsv_path is None:
    print("ERROR: No TSV file found in dataset directory.")
    df = None
else:
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"TSV path:   {tsv_path}")
    print(f"Rows:       {len(df)}")
    print(f"Columns:    {list(df.columns)}")
    print("\n=== First 5 rows ===")
    display(df.head())
    print("\n=== Column dtypes ===")
    print(df.dtypes)
    print("\n=== Null counts ===")
    print(df.isnull().sum())

In [ ]:
# Cell 2.3 — Annotation analysis (CRITICAL)
from pathlib import Path
import re

root = Path("data/simongraves-license-plate-dataset")

SPATIAL_PATTERNS = [
    "bbox", "bounding_box", "boundingbox",
    "x1", "y1", "x2", "y2",
    "xmin", "ymin", "xmax", "ymax",
    "polygon", "mask", "segmentation",
    "coordinates", "region", "annotation", "points",
]

ANNOTATION_TYPE = None  # set below; used by later sections

if df is None:
    print("ERROR: No TSV dataframe available. Re-run Cell 2.2.")
else:
    cols_lower = {c: c.lower().replace(" ", "_") for c in df.columns}
    spatial_hits = []
    for col, low in cols_lower.items():
        if any(pat in low for pat in SPATIAL_PATTERNS):
            spatial_hits.append(col)

    print("=== Columns matching spatial patterns ===")
    if not spatial_hits:
        print("  (none)")
    for col in spatial_hits:
        series = df[col]
        first = series.dropna().iloc[0] if series.notna().any() else None
        print(f"  {col!r}: first non-null = {first!r}")

    mask_dirs = [p for p in root.rglob("*") if p.is_dir() and "mask" in p.name.lower()]
    mask_pngs = list(root.rglob("*mask*.png")) + list(root.rglob("*mask*.jpg"))
    has_raster = bool(mask_dirs) or bool(mask_pngs)

    has_polygon = any(
        re.search(r"polygon|segmentation|points|coordinates", c.lower())
        for c in spatial_hits
    )
    has_bbox = any(
        re.search(
            r"bbox|bounding|xmin|xmax|ymin|ymax|^x1$|^y1$|^x2$|^y2$",
            c.lower().replace(" ", "_"),
        )
        for c in spatial_hits
    )

    print("\n=== Heuristic flags ===")
    print(f"  raster mask dirs/files: {has_raster}")
    print(f"  polygon-like columns:   {has_polygon}")
    print(f"  bbox-like columns:      {has_bbox}")
    print(f"  spatial hits:           {spatial_hits}")

    if has_raster:
        ANNOTATION_TYPE = "A"
        print("\nANNOTATION TYPE: RASTER MASKS FOUND       → Case A (best case)")
    elif has_polygon:
        ANNOTATION_TYPE = "B"
        print("\nANNOTATION TYPE: POLYGON COORDINATES      → Case B (requires rasterization)")
    elif has_bbox:
        ANNOTATION_TYPE = "C"
        print("\nANNOTATION TYPE: BOUNDING BOX ONLY        → Case C (requires rasterization, less precise)")
    else:
        ANNOTATION_TYPE = "D"
        print("\nANNOTATION TYPE: OCR/TEXT ONLY            → Case D (CANNOT TRAIN SEGMENTATION — STOP)")
        print("\n" + "=" * 72)
        print("ERROR: This dataset provides no spatial masks, polygons, or bounding boxes.")
        print("Segmentation training is NOT possible with OCR/text labels alone.")
        print("Do NOT run Section 3 (mask generation) or training sections.")
        print("Switch to a dataset with pixel/polygon/bbox annotations (e.g. TLPD).")
        print("=" * 72)

    print(f"\nANNOTATION_TYPE variable = {ANNOTATION_TYPE!r}")

In [ ]:
# Cell 2.4 — Image sample inspection
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("data/simongraves-license-plate-dataset")

image_paths = list(root.rglob("*.jpg")) + list(root.rglob("*.png"))
image_paths = [p for p in image_paths if "masks" not in str(p).lower()]
image_paths = image_paths[:3]

if not image_paths:
    print("ERROR: No JPG/PNG images found under the dataset root.")
else:
    fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 5))
    if len(image_paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, image_paths):
        img = cv2.imread(str(p))
        if img is None:
            ax.set_title(f"FAILED: {p.name}")
            ax.axis("off")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        print(f"path={p}")
        print(f"  shape={img.shape}  dtype={img.dtype}  channels={img.shape[2] if img.ndim == 3 else 1}")
        ax.imshow(img_rgb)
        ax.set_title(f"{p.name}\n{img.shape}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

✅ **STOP — Section 2 complete.**

Review the annotation type reported above (**Case A / B / C / D**).

**Do NOT run Section 3 until you understand and approve the annotation type.**

If **Case D** was detected, training is not possible without a different dataset.

## SECTION 3 — Mask Generation (conditional)

⚠️ **This section applies ONLY if Section 2 detected bounding-box or polygon annotations (Case B or C).**

- If raster masks already exist (**Case A**), skip to Section 4.
- If only OCR text was found (**Case D**), do **NOT** run this section.

In [ ]:
# Cell 3.1 — Generate binary masks from bounding boxes (Case C)
# This cell runs only if Case C (bounding box) was confirmed in Section 2.
# Adjust column names to match actual TSV columns from Cell 2.2.

import cv2
import numpy as np
from pathlib import Path

assert "ANNOTATION_TYPE" in dir() and ANNOTATION_TYPE is not None, "Run Section 2 Cell 2.3 first."

if ANNOTATION_TYPE == "D":
    print("SKIPPED: Case D (OCR/text only). Mask generation is not possible.")
elif ANNOTATION_TYPE == "A":
    print("SKIPPED: Case A (raster masks already present). Go to Section 4.")
elif ANNOTATION_TYPE == "B":
    print("Case B detected (polygon). Bounding-box rasterization below is for Case C.")
    print("You must adapt this cell to fill polygons (e.g. cv2.fillPoly) after approving coordinates.")
elif ANNOTATION_TYPE == "C":
    MASK_DIR = Path("data/simongraves-license-plate-dataset/masks")
    MASK_DIR.mkdir(parents=True, exist_ok=True)

    # ---- ADJUST THESE COLUMN NAMES after inspecting Cell 2.2 output ----
    FILENAME_COL = "filename"  # column with image filename
    BBOX_COL = None  # set to column name if single bbox string like "x1,y1,x2,y2"
    X1_COL = None
    Y1_COL = None
    X2_COL = None
    Y2_COL = None
    # -------------------------------------------------------------------

    if BBOX_COL is None and any(c is None for c in (X1_COL, Y1_COL, X2_COL, Y2_COL)):
        print("ERROR: Set BBOX_COL or X1/Y1/X2/Y2 column names before generating masks.")
        print(f"Available columns: {list(df.columns)}")
    else:
        generated = 0
        skipped = 0

        for _, row in df.iterrows():
            img_path = next(root.rglob(str(row[FILENAME_COL])), None)
            if img_path is None:
                skipped += 1
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                skipped += 1
                continue

            h, w = img.shape[:2]
            mask = np.zeros((h, w), dtype=np.uint8)

            if BBOX_COL:
                coords = list(map(int, str(row[BBOX_COL]).split(",")))
                x1, y1, x2, y2 = coords
            else:
                x1 = int(row[X1_COL])
                y1 = int(row[Y1_COL])
                x2 = int(row[X2_COL])
                y2 = int(row[Y2_COL])

            x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
            y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
            mask[y1:y2, x1:x2] = 255

            mask_path = MASK_DIR / (img_path.stem + ".png")
            cv2.imwrite(str(mask_path), mask)
            generated += 1

        print(f"Masks generated: {generated}")
        print(f"Skipped (not found / unreadable): {skipped}")
        print(f"Mask directory: {MASK_DIR.resolve()}")
else:
    print(f"Unknown ANNOTATION_TYPE={ANNOTATION_TYPE!r}")

In [ ]:
# Cell 3.2 — Verify generated masks (run after Case B/C mask generation)
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

root = Path("data/simongraves-license-plate-dataset")
MASK_DIR = Path("data/simongraves-license-plate-dataset/masks")

if not MASK_DIR.is_dir() or not any(MASK_DIR.glob("*.png")):
    print("No masks found under data/simongraves-license-plate-dataset/masks/")
    print("Skip this cell if you are on Case A (existing masks) or Case D (no spatial labels).")
else:
    mask_paths = list(MASK_DIR.glob("*.png"))[:5]
    fig, axes = plt.subplots(len(mask_paths), 3, figsize=(15, 4 * len(mask_paths)))
    if len(mask_paths) == 1:
        axes = axes.reshape(1, 3)

    for i, mp in enumerate(mask_paths):
        candidates = [p for p in root.rglob(mp.stem + ".*") if "masks" not in str(p)]
        img_path = candidates[0] if candidates else None
        if img_path is None:
            print(f"No image for mask {mp.name}")
            continue

        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)

        overlay = img.copy()
        overlay[mask > 0] = [255, 0, 0]
        blended = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Image")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title("Mask")
        axes[i, 1].axis("off")
        axes[i, 2].imshow(blended)
        axes[i, 2].set_title("Overlay")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()

✅ **STOP — Section 3 complete.**

Examine the overlay visualizations carefully.

Confirm that the red mask region correctly covers the license plate in each image.

If masks look wrong, do NOT proceed to training. Investigate the coordinate parsing above.

## SECTION 4 — Dataset Validation

Statistics on all image–mask pairs before building the DataLoader.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from collections import Counter

root = Path("data/simongraves-license-plate-dataset")
MASK_DIR = Path("data/simongraves-license-plate-dataset/masks")

image_paths = [
    p for p in root.rglob("*")
    if p.suffix.lower() in (".jpg", ".jpeg", ".png") and "masks" not in str(p)
]
mask_paths = list(MASK_DIR.glob("*.png")) if MASK_DIR.is_dir() else []
mask_stems = {p.stem for p in mask_paths}

matched = []
missing_mask = []
corrupted = []
fg_ratios = []
shapes = []

for img_p in image_paths:
    if img_p.stem not in mask_stems:
        missing_mask.append(img_p)
        continue
    img = cv2.imread(str(img_p))
    if img is None:
        corrupted.append(img_p)
        continue
    msk = cv2.imread(str(MASK_DIR / (img_p.stem + ".png")), cv2.IMREAD_GRAYSCALE)
    if msk is None:
        corrupted.append(img_p)
        continue
    matched.append((img_p, MASK_DIR / (img_p.stem + ".png")))
    fg = (msk > 0).sum() / msk.size
    fg_ratios.append(fg)
    shapes.append(img.shape[:2])

print(f"Total images:        {len(image_paths)}")
print(f"Total masks:         {len(mask_paths)}")
print(f"Matched pairs:       {len(matched)}")
print(f"Missing masks:       {len(missing_mask)}")
print(f"Corrupted files:     {len(corrupted)}")
if fg_ratios:
    print(f"Avg foreground %:    {np.mean(fg_ratios)*100:.2f}%")
    print(f"Min foreground %:    {np.min(fg_ratios)*100:.2f}%")
    print(f"Max foreground %:    {np.max(fg_pcts)*100:.2f}%")
    sample_mask = cv2.imread(str(matched[0][1]), cv2.IMREAD_GRAYSCALE)
    print(f"Sample mask unique:  {np.unique(sample_mask)}")
if shapes:
    shape_counter = Counter(shapes)
    print("\nTop image sizes:")
    for shape, count in shape_counter.most_common(5):
        print(f"  {shape}: {count} images")

if len(matched) == 0:
    print("\nWARNING: No matched image-mask pairs. Do not proceed to training.")

✅ **STOP — Section 4 complete.**

Confirm matched pairs > 0 before continuing.

If foreground % is extremely low or 0, masks are likely wrong. Do not train.

## SECTION 5 — Dataset Class and DataLoader

Train/validation split (80/20), Albumentations, and a PyTorch `Dataset`.

In [ ]:
# Cell 5.1 — Train/Validation split
import random

random.seed(42)

all_pairs = matched  # from Section 4

if not all_pairs:
    raise RuntimeError("No matched pairs from Section 4. Cannot build train/val split.")

random.shuffle(all_pairs)
split_idx = int(0.8 * len(all_pairs))
train_pairs = all_pairs[:split_idx]
val_pairs = all_pairs[split_idx:]

print(f"Training samples:   {len(train_pairs)}")
print(f"Validation samples: {len(val_pairs)}")

In [ ]:
# Cell 5.2 — Dataset class
import random
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

random.seed(42)
torch.manual_seed(42)

IMG_SIZE = 512  # reduce to 384 or 256 if VRAM OOM

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class LicensePlateDataset(Dataset):
    def __init__(self, pairs, transforms=None):
        self.pairs = pairs
        self.transforms = transforms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, msk_path = self.pairs[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)  # binary 0/1

        if self.transforms:
            out = self.transforms(image=image, mask=mask)
            image = out["image"]
            mask = out["mask"].unsqueeze(0)  # (1, H, W)

        return image, mask


print(f"IMG_SIZE={IMG_SIZE}")
print("LicensePlateDataset defined.")

In [ ]:
# Cell 5.3 — DataLoader
from torch.utils.data import DataLoader

BATCH_SIZE = 4  # Safe for RTX 3070 8GB at 512×512
NUM_WORKERS = 0  # Windows: keep at 0 to avoid multiprocessing issues

train_ds = LicensePlateDataset(train_pairs, transforms=train_transforms)
val_ds = LicensePlateDataset(val_pairs, transforms=val_transforms)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

# Sanity check
images, masks = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")  # expect (B, 3, 512, 512)
print(f"Mask batch shape:  {masks.shape}")  # expect (B, 1, 512, 512)
print(f"Mask unique values: {masks.unique()}")

✅ **STOP — Section 5 complete.**

Confirm image batch shape is `(B, 3, H, W)` and mask is `(B, 1, H, W)`.

Confirm mask unique values are `[0.0, 1.0]`.

## SECTION 6 — Model Architecture

U-Net + ResNet34 (ImageNet), 3 input channels, 1 output channel (logits).

In [ ]:
import segmentation_models_pytorch as smp
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,  # raw logits — BCEWithLogitsLoss handles sigmoid
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass test
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
dummy_output = model(dummy_input)
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")  # expect (1, 1, 512, 512)

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**2:.1f} MB")

✅ **STOP — Section 6 complete.**

Confirm output shape is `(1, 1, H, W)` and no OOM error occurred.

## SECTION 7 — Loss, Optimizer, and Metrics

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# Combined Dice + BCE loss
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)


# Metrics
def compute_metrics(pred_logits, target, threshold=0.5):
    pred_probs = torch.sigmoid(pred_logits)
    pred_bin = (pred_probs > threshold).float()
    target = target.float()

    TP = (pred_bin * target).sum()
    FP = (pred_bin * (1 - target)).sum()
    FN = ((1 - pred_bin) * target).sum()
    TN = ((1 - pred_bin) * (1 - target)).sum()

    dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    iou = TP / (TP + FP + FN + 1e-8)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    return {
        "dice": dice.item(),
        "iou": iou.item(),
        "precision": precision.item(),
        "recall": recall.item(),
    }


# Optimizer and scheduler
LEARNING_RATE = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

print("Loss function: 0.5 × BCE + 0.5 × Dice")
print(f"Optimizer:     AdamW  lr={LEARNING_RATE}")
print("Scheduler:     CosineAnnealingLR  T_max=30")

✅ **STOP — Section 7 complete.**

Review the loss function and optimizer configuration.

Approve before running Section 8 (sanity check) and Section 9 (full training).

## SECTION 8 — Sanity Check (2 mini-epochs)

Short train loop to verify loss is finite and VRAM is safe.

In [ ]:
import torch
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA required for the approved training setup.")

SANITY_EPOCHS = 2
scaler = torch.amp.GradScaler("cuda")  # AMP / mixed precision

model.train()
for epoch in range(SANITY_EPOCHS):
    total_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Sanity epoch {epoch+1}"):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            preds = model(images)
            loss = combined_loss(preds, masks)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        vram_used = torch.cuda.memory_allocated() / 1024**2

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | VRAM: {vram_used:.0f} MB")

print("\nSanity check complete. No OOM errors, loss is finite.")
print("Ready for full training if loss decreased and VRAM < 7500 MB.")

✅ **STOP — Section 8 (Sanity Check) complete.**

Before running full training, confirm:

- Loss decreased between epoch 1 and 2
- Loss is NOT NaN or Inf
- VRAM usage is below 7500 MB
- No CUDA out-of-memory errors

If VRAM exceeded 7500 MB, reduce `BATCH_SIZE` or `IMG_SIZE` and re-run.

Only run Section 9 after these checks pass.

## SECTION 9 — Full Training Loop

⚠️ **Do not run this section until the sanity check in Section 8 passed.**

In [ ]:
import torch
import json
from pathlib import Path
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA required for full training.")

NUM_EPOCHS = 50
CKPT_DIR = Path("checkpoints")
RESULTS_DIR = Path("results")
CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

scaler = torch.amp.GradScaler("cuda")
best_dice = 0.0
history = {
    "train_loss": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": [],
    "val_precision": [],
    "val_recall": [],
}

for epoch in range(1, NUM_EPOCHS + 1):
    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [TRAIN]", leave=False):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            preds = model(images)
            loss = combined_loss(preds, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---------- VALIDATE ----------
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0
    val_prec = 0.0
    val_rec = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [VAL]", leave=False):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            with torch.amp.autocast("cuda"):
                preds = model(images)
                loss = combined_loss(preds, masks)

            metrics = compute_metrics(preds, masks)
            val_loss += loss.item()
            val_dice += metrics["dice"]
            val_iou += metrics["iou"]
            val_prec += metrics["precision"]
            val_rec += metrics["recall"]

    n = len(val_loader)
    val_loss /= n
    val_dice /= n
    val_iou /= n
    val_prec /= n
    val_rec /= n

    scheduler.step()

    # ---------- LOG ----------
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)
    history["val_precision"].append(val_prec)
    history["val_recall"].append(val_rec)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Prec: {val_prec:.4f} | "
        f"Rec: {val_rec:.4f} | "
        f"VRAM: {torch.cuda.memory_allocated()/1024**2:.0f} MB"
    )

    # ---------- CHECKPOINT ----------
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_dice": val_dice,
            "val_iou": val_iou,
        },
        CKPT_DIR / "latest_checkpoint.pth",
    )

    # Save best Dice only when improved (never overwrite with a worse checkpoint)
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_dice,
                "val_iou": val_iou,
            },
            CKPT_DIR / "best_model.pth",
        )
        print(f"  *** New best model saved at epoch {epoch} | Dice: {best_dice:.4f} ***")

    with open(RESULTS_DIR / "training_log.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

print(f"\nTraining complete. Best Dice: {best_dice:.4f}")

✅ **STOP — Section 9 complete.**

Confirm `checkpoints/best_model.pth` and `results/training_log.json` exist before plotting curves.

## SECTION 10 — Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

with open(Path("results/training_log.json"), encoding="utf-8") as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_dice"], label="Dice", color="green")
axes[1].plot(history["val_iou"], label="IoU", color="blue")
axes[1].set_title("Dice & IoU")
axes[1].legend()
axes[1].set_xlabel("Epoch")

axes[2].plot(history["val_precision"], label="Precision", color="orange")
axes[2].plot(history["val_recall"], label="Recall", color="red")
axes[2].set_title("Precision & Recall")
axes[2].legend()
axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()
print("Training curves saved to: results/training_curves.png")

✅ **STOP — Section 10 complete.**

Review the curves for overfitting or unstable loss before evaluation.

## SECTION 11 — Model Evaluation

Load the best checkpoint and evaluate on the validation set. Save sample visualizations.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = Path("checkpoints")
RESULTS_DIR = Path("results/predictions")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ckpt = torch.load(CKPT_DIR / "best_model.pth", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} | Val Dice: {ckpt['val_dice']:.4f}")

INV_MEAN = np.array([0.485, 0.456, 0.406])
INV_STD = np.array([0.229, 0.224, 0.225])


def denormalize(tensor):
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * INV_STD + INV_MEAN
    return np.clip(img, 0, 1)


total_dice = 0.0
total_iou = 0.0
total_prec = 0.0
total_rec = 0.0
n_batches = 0
sample_idx = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            preds = model(images)

        metrics = compute_metrics(preds, masks)
        total_dice += metrics["dice"]
        total_iou += metrics["iou"]
        total_prec += metrics["precision"]
        total_rec += metrics["recall"]
        n_batches += 1

        pred_probs = torch.sigmoid(preds)
        pred_bin = (pred_probs > 0.5).float()

        for i in range(images.shape[0]):
            if sample_idx >= 10:
                break

            img_np = denormalize(images[i])
            gt_np = masks[i, 0].cpu().numpy()
            pred_np = pred_bin[i, 0].cpu().numpy()

            gt_overlay = (img_np * 255).astype(np.uint8).copy()
            pred_overlay = (img_np * 255).astype(np.uint8).copy()
            gt_overlay[gt_np > 0] = [255, 0, 0]
            pred_overlay[pred_np > 0] = [0, 255, 0]

            fig, axes = plt.subplots(1, 5, figsize=(20, 4))
            axes[0].imshow(img_np)
            axes[0].set_title("Image")
            axes[0].axis("off")
            axes[1].imshow(gt_np, cmap="gray")
            axes[1].set_title("GT Mask")
            axes[1].axis("off")
            axes[2].imshow(pred_np, cmap="gray")
            axes[2].set_title("Pred Mask")
            axes[2].axis("off")
            axes[3].imshow(gt_overlay)
            axes[3].set_title("GT Overlay")
            axes[3].axis("off")
            axes[4].imshow(pred_overlay)
            axes[4].set_title("Pred Overlay")
            axes[4].axis("off")

            plt.tight_layout()
            save_path = RESULTS_DIR / f"sample_{sample_idx:03d}.png"
            plt.savefig(save_path, dpi=100)
            plt.close()
            sample_idx += 1

print("\n=== FINAL EVALUATION RESULTS ===")
print(f"Dice:      {total_dice / n_batches:.4f}")
print(f"IoU:       {total_iou  / n_batches:.4f}")
print(f"Precision: {total_prec / n_batches:.4f}")
print(f"Recall:    {total_rec  / n_batches:.4f}")
print(f"\nPrediction visualizations saved to: {RESULTS_DIR.resolve()}")

✅ **STOP — Section 11 complete.**

Fill in the Final Report template in Section 12 with the printed metrics.

## SECTION 12 — Final Report

Fill in bracketed fields after you finish Sections 2–11.

---

## FINAL REPORT

### Environment
| Item | Value |
|------|-------|
| Python | 3.12.10 |
| PyTorch | 2.6.0+cu124 |
| CUDA | 12.4 |
| GPU | NVIDIA GeForce RTX 3070 |
| VRAM | 8 GB |

### Dataset
| Item | Value |
|------|-------|
| Dataset | Simon Graves License Plate Dataset |
| Kaggle | https://www.kaggle.com/datasets/simongraves/license-plate-dataset |
| Local path | data/simongraves-license-plate-dataset/ |
| Total images | [fill after Section 4] |
| Train samples | [fill after Section 5] |
| Val samples | [fill after Section 5] |
| Annotation type | [fill after Section 2 — Case A/B/C/D] |
| Mask type | Binary (0 = background, 255 = plate) |

### Model
| Item | Value |
|------|-------|
| Architecture | U-Net |
| Encoder | resnet34 (ImageNet pretrained) |
| Parameters | ~24M |
| Input resolution | 512×512 |
| Loss | 0.5 × BCE + 0.5 × Dice |

### Training
| Item | Value |
|------|-------|
| Epochs | 50 |
| Batch size | 4 |
| Learning rate | 1e-4 |
| Optimizer | AdamW |
| Scheduler | CosineAnnealingLR |
| AMP / FP16 | Yes |

### Results
| Metric | Value |
|--------|-------|
| Best Val Dice | [fill after Section 9] |
| IoU | [fill after Section 11] |
| Precision | [fill after Section 11] |
| Recall | [fill after Section 11] |
| Best epoch | [fill after Section 9] |

### Generated Files
| File | Path |
|------|------|
| Best checkpoint | checkpoints/best_model.pth |
| Latest checkpoint | checkpoints/latest_checkpoint.pth |
| Training log | results/training_log.json |
| Training curves | results/training_curves.png |
| Prediction samples | results/predictions/sample_*.png |

---

**Note:** If Section 2 reported **Case D (OCR/text only)**, do not train on this dataset. Use a dataset with spatial annotations instead.